# main_pipeline (Kaggle)

Orchestrator chạy toàn bộ pipeline (build data -> train -> chat demo -> evaluate)
trên Kaggle (có GPU free).

**Chuẩn bị trước khi chạy:**
1. Upload cả thư mục `medquad_rag/` (chứa `src/`, `pipeline/`, `requirements.txt`)
   làm 1 Kaggle Dataset, rồi add vào notebook này (Add Input), HOẶC `!git clone`
   repo nếu đã đẩy lên GitHub.
2. Upload `final_train_dataset.json` vào `data/` của project (hoặc add làm Dataset riêng
   rồi copy vào `data/`).
3. Sửa biến `PROJECT_DIR` ở cell dưới cho đúng đường dẫn sau khi add input.

In [ ]:
pip install -q --force-reinstall --no-cache-dir "numpy==1.26.4" "pandas==2.2.2" "pyarrow>=14,<18"

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_XET"] = "1"

In [2]:
!git clone https://github.com/nguyentuanphat2594/test_mockproject.git

Cloning into 'test_mockproject'...
remote: Enumerating objects: 11394, done.
remote: Counting objects: 100% (11394/11394), done.
remote: Compressing objects: 100% (4577/4577), done.
remote: Total 11394 (delta 6810), reused 11389 (delta 6805), pack-reused 0 (from 0)
Receiving objects: 100% (11394/11394), 27.10 MiB | 24.80 MiB/s, done.
Resolving deltas: 100% (6810/6810), done.


In [3]:
PROJECT_DIR = "/kaggle/working/test_mockproject/test_train_again/Training"

import sys
sys.path.insert(0, PROJECT_DIR)

%cd /kaggle/working/test_mockproject/test_train_again/Training

/kaggle/working/test_mockproject/test_train_again/Training


In [4]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 92.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.9/503.9 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 31.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.4/396.4 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [5]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
print("ragas import OK")

ragas import OK


## Bước 1: Build train.jsonl từ final_train_dataset.json

In [6]:
from pipeline import build_train_dataset
build_train_dataset.main()

Total samples (raw)   : 11548
Sample limit áp dụng  : 11548
Hợp lệ (validated)    : 11548
Bỏ qua (thiếu Q/A)    : 0
USE_RAG               : False
----------------------------------------
Train : 8083 -> /kaggle/working/test_mockproject/test_train_again/Training/output/train.jsonl
Val   : 1732 -> /kaggle/working/test_mockproject/test_train_again/Training/output/val.jsonl
Test  : 1733 -> /kaggle/working/test_mockproject/test_train_again/Training/output/test.jsonl


## Bước 2: Train LoRA

In [7]:
from pipeline import train
train.main()

Loading model + tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

{'eval_loss': 0.877522885799408, 'eval_runtime': 145.9917, 'eval_samples_per_second': 11.864, 'eval_steps_per_second': 5.932, 'eval_num_tokens': 2394633.0, 'eval_mean_token_accuracy': 0.8000149626379476, 'epoch': 0.89}
{'loss': 1.0956, 'grad_norm': 0.6785995960235596, 'learning_rate': 2.195845697329377e-05, 'num_tokens': 2397285.0, 'mean_token_accuracy': 0.7566842287778854, 'epoch': 0.89}
{'loss': 0.8562, 'grad_norm': 0.6607750058174133, 'learning_rate': 2.176063303659743e-05, 'num_tokens': 2399727.0, 'mean_token_accuracy': 0.7831371128559113, 'epoch': 0.89}
{'loss': 0.7598, 'grad_norm': 0.7364756464958191, 'learning_rate': 2.156280909990109e-05, 'num_tokens': 2401769.0, 'mean_token_accuracy': 0.8228389173746109, 'epoch': 0.89}
{'loss': 0.8142, 'grad_norm': 0.6336453557014465, 'learning_rate': 2.1364985163204748e-05, 'num_tokens': 2405358.0, 'mean_token_accuracy': 0.8104185312986374, 'epoch': 0.89}
{'loss': 0.8501, 'grad_norm': 0.6878100633621216, 'learning_rate': 2.1167161226508407e-0

  0%|          | 0/866 [00:00<?, ?it/s]

{'eval_loss': 0.876058042049408, 'eval_runtime': 146.9273, 'eval_samples_per_second': 11.788, 'eval_steps_per_second': 5.894, 'eval_num_tokens': 2520063.0, 'eval_mean_token_accuracy': 0.8004413178280925, 'epoch': 0.94}
{'loss': 0.5094, 'grad_norm': 0.6331624388694763, 'learning_rate': 1.2067260138476756e-05, 'num_tokens': 2522008.0, 'mean_token_accuracy': 0.8683823049068451, 'epoch': 0.94}
{'loss': 1.0762, 'grad_norm': 0.6886830925941467, 'learning_rate': 1.1869436201780416e-05, 'num_tokens': 2525114.0, 'mean_token_accuracy': 0.7672040611505508, 'epoch': 0.94}
{'loss': 1.2504, 'grad_norm': 0.7215785980224609, 'learning_rate': 1.1671612265084075e-05, 'num_tokens': 2527540.0, 'mean_token_accuracy': 0.7282232493162155, 'epoch': 0.94}
{'loss': 0.9147, 'grad_norm': 0.6186951994895935, 'learning_rate': 1.1473788328387735e-05, 'num_tokens': 2530482.0, 'mean_token_accuracy': 0.7846471816301346, 'epoch': 0.94}
{'loss': 0.7909, 'grad_norm': 0.6138326525688171, 'learning_rate': 1.1275964391691394

  0%|          | 0/866 [00:00<?, ?it/s]

{'eval_loss': 0.8756227493286133, 'eval_runtime': 145.9526, 'eval_samples_per_second': 11.867, 'eval_steps_per_second': 5.933, 'eval_num_tokens': 2659805.0, 'eval_mean_token_accuracy': 0.8004081337336304, 'epoch': 0.99}
{'loss': 0.804, 'grad_norm': 0.5393854379653931, 'learning_rate': 2.1760633036597427e-06, 'num_tokens': 2663234.0, 'mean_token_accuracy': 0.8050128817558289, 'epoch': 0.99}
{'loss': 0.9983, 'grad_norm': 0.5965462923049927, 'learning_rate': 1.9782393669634024e-06, 'num_tokens': 2666795.0, 'mean_token_accuracy': 0.7615407556295395, 'epoch': 0.99}
{'loss': 0.7432, 'grad_norm': 0.6651788949966431, 'learning_rate': 1.7804154302670625e-06, 'num_tokens': 2669023.0, 'mean_token_accuracy': 0.8193870782852173, 'epoch': 0.99}
{'loss': 1.013, 'grad_norm': 0.6667025089263916, 'learning_rate': 1.5825914935707222e-06, 'num_tokens': 2672855.0, 'mean_token_accuracy': 0.7664442211389542, 'epoch': 0.99}
{'loss': 0.8392, 'grad_norm': 0.7988487482070923, 'learning_rate': 1.3847675568743818e

## Bước 3: Demo chat (inference)

In [ ]:
from pipeline import chat
chat.main()

## Bước 4: Đánh giá bằng RAGAs

In [ ]:
from pipeline import evaluate
evaluate.main()

## (Tuỳ chọn) Lưu output_model về lại Kaggle Output

`output/output_model/` đã nằm trong `/kaggle/working/`, Kaggle sẽ tự lưu làm
Output của notebook này khi Save Version — không cần thêm gì.

In [9]:
%cd /kaggle/working/test_mockproject/test_train_again/Training
!zip -r output.zip output

/kaggle/working/test_mockproject/test_train_again/Training
updating: output/ (stored 0%)
updating: output/training_summary.csv (deflated 29%)
updating: output/.gitkeep (stored 0%)
updating: output/output_model/ (stored 0%)
updating: output/output_model/checkpoint-1011/ (stored 0%)
updating: output/output_model/checkpoint-1011/optimizer.pt

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 8%)
updating: output/output_model/checkpoint-1011/added_tokens.json (deflated 67%)
updating: output/output_model/checkpoint-1011/training_args.bin (deflated 53%)
updating: output/output_model/checkpoint-1011/README.md (deflated 65%)
updating: output/output_model/checkpoint-1011/merges.txt (deflated 57%)
updating: output/output_model/checkpoint-1011/trainer_state.json (deflated 80%)
updating: output/output_model/checkpoint-1011/adapter_model.safetensors (deflated 7%)
updating: output/output_model/checkpoint-1011/special_tokens_map.json (deflated 69%)
updating: output/output_model/checkpoint-1011/chat_template.jinja (deflated 71%)
updating: output/output_model/checkpoint-1011/vocab.json (deflated 61%)
updating: output/output_model/checkpoint-1011/scheduler.pt (deflated 61%)
updating: output/output_model/checkpoint-1011/tokenizer_config.json (deflated 89%)
updating: output/output_model/checkpoint-1011/adapter_config.json (deflated 56%)
updating: output/output_model/checkpoint-1